# Token-weighted gradient accumulation with autograd

Language-model losses are a mean over **tokens**, not over sequences. When you split a padded batch into micro-batches and call `backward()` on each, PyTorch adds the gradients into `.grad`, so each micro-batch loss must be scaled to make the sum equal the full-batch gradient. The common shortcut, `loss_k.mean() / n_micro`, is only right when every micro-batch has the same number of real tokens.

Write `accumulate_grads(X, targets, W, b, n_micro)` for a linear next-token head.

## Inputs
- `X`: `float64` `(B, T, D)` hidden states. `W`: `float64` `(D, V)`. `b`: `float64` `(V,)`.
- `targets`: `int64` `(B, T)`, token ids in $[0, V)$, or `-100` for padding (ignored).
- `n_micro`: int, $1 \le$ `n_micro` $\le B$. Split the batch along dimension 0 with `torch.tensor_split(..., n_micro)` (contiguous chunks, earlier chunks one larger when $B$ is not divisible).

The full-batch loss is the mean cross-entropy over all real tokens of the batch:
$$\mathcal L = \frac{1}{N_\text{tok}} \sum_{(i,t)\,:\,y_{it} \ne -100} -\log \operatorname{softmax}(x_{it} W + b)_{y_{it}}$$

The batch has at least one real token, but a micro-batch may have none.

## What to compute
1. **Correct accumulation**: for each micro-batch $k$, compute its summed token loss, divide by the **full-batch** token count $N_\text{tok}$, and call `backward()`, accumulating into `.grad` of leaf copies of `W` and `b` (gradients start at zero).
2. **Naive accumulation**: for each micro-batch with at least one real token, call `backward()` on (its mean token loss) $/\,$`n_micro`. Skip micro-batches with no real tokens (their mean would be NaN), but still divide by `n_micro`.

## Output
A tuple `(gW, gb, naive_gW, naive_gb)` of `float64` tensors shaped like `W`, `b`, `W`, `b`. Do not modify the inputs.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import torch.nn.functional as F

def accumulate_grads(X, targets, W, b, n_micro):
    """Return (gW, gb, naive_gW, naive_gb) from micro-batch backward passes."""
    W = W.clone().requires_grad_(True)
    b = b.clone().requires_grad_(True)
    for Xk, yk in zip(torch.tensor_split(X, n_micro), torch.tensor_split(targets, n_micro)):
        logits = Xk @ W + b
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), yk.reshape(-1), ignore_index=-100)
        (loss / n_micro).backward()          # TODO: right only if all micro-batches have equal token counts
    return W.grad, b.grad, W.grad, b.grad


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[0.345584192064786, 0.8216181435011584], [0.33043707618338714, -1.303157231604361], [0.9053558666731177, 0.4463745723640113]], [[-0.5369532353602852, 0.5811181041963531], [0.36457239618607573, 0.294132496655526], [0.02842224131579679, 0.5467129866124469]]], "dtype": "float64"}, {"__nd__": [[2, 0, 1], [2, -100, -100]], "dtype": "int64"}, {"__nd__": [[0.2994231063173138, 0.019861053740829495, -0.1462283754825443], [-0.39095423117842104, -0.12859612030943535, 0.004071090259171754]], "dtype": "float64"}, {"__nd__": [-0.027560290529937043, 0.12940638143982072, 0.10067243153057943], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[0.023017747176460275, -0.13791568838586843, 0.11489794120940816], [0.31398471615479295, -0.051870216061420005, -0.2621145000933729]], "dtype": "float64"}, {"__nd__": [0.06706904739899576, 0.0978632774562538, -0.1649323248552495], "dtype": "float64"}, {"__nd__": [[-0.025331666312848986, -0.15634923463039777, 0.1816809009432467], [0.25334567980136236, 0.03512270176964005, -0.2884683815710024]], "dtype": "float64"}, {"__nd__": [0.1204675842010125, 0.18518827735248866, -0.3056558615535011], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.345584192064786, 0.8216181435011584], [0.33043707618338714, -1.303157231604361], [0.9053558666731177, 0.4463745723640113]], [[-0.5369532353602852, 0.5811181041963531], [0.36457239618607573, 0.294132496655526], [0.02842224131579679, 0.5467129866124469]]], "dtype": "float64"}, {"__nd__": [[2, 0, 1], [2, -100, -100]], "dtype": "int64"}, {"__nd__": [[0.2994231063173138, 0.019861053740829495, -0.1462283754825443], [-0.39095423117842104, -0.12859612030943535, 0.004071090259171754]], "dtype": "float64"}, {"__nd__": [-0.027560290529937043, 0.12940638143982072, 0.10067243153057943], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[0.023017747176460275, -0.13791568838586843, 0.11489794120940816], [0.31398471615479295, -0.051870216061420005, -0.26211450009337295]], "dtype": "float64"}, {"__nd__": [0.06706904739899576, 0.0978632774562538, -0.1649323248552495], "dtype": "float64"}, {"__nd__": [[0.023017747176460275, -0.13791568838586843, 0.11489794120940816], [0.31398471615479295, -0.051870216061420005, -0.26211450009337295]], "dtype": "float64"}, {"__nd__": [0.06706904739899576, 0.0978632774562538, -0.1649323248552495], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.18905338179353307, -0.5227484414807474, -0.41306354339189344], [-2.4414673826398556, 1.799707382720902, 1.1441658720372287], [-0.32542283686782436, 0.7738065867276614, 0.28121066979764925], [-0.5538228364240524, 0.9775674511260357, -0.31055654665915255], [-0.3288239040579627, -0.7921467553588982, 0.45495807124085547]], [[-0.09919805171738795, 0.5452887139646817, -0.6071856998706371], [0.12682784711186987, -0.8922740434297903, 0.8414649723701431], [0.18803508698068597, 0.33057100813532614, 0.41050391297026284], [-1.0107575001533344, 0.7831809961440773, 2.0567028183423686], [-1.6384425032355252, -1.7294114671544816, -1.50483141386432]], [[0.8414588934539998, 0.12871565747406846, 1.078342440739298], [0.722430872307499, 0.21057181237528058, 0.28403814525037085], [-0.16976049772313542, 0.8684602112115102, -1.1297159617807548], [-0.4218588261783162, 0.2429388530987352, 1.8014208584493328], [-0.7644641157203993, -1.0790604591369424, -0.5632871985040379]], [[0.9692722011381628, -0.23500550689182578, 1.324347019236957], [-1.872529517606024, 1.128523141900478, 1.0348662399295387], [-1.4186656898014849, 0.15358188653595775, 1.2157576351945363], [0.08791904230952238, 0.99970142215178, 2.3748869133171437], [0.2739322545923299, -0.2803823139429379, -0.7710521598195307]]], "dtype": "float64"}, {"__nd__": [[3, 0, 1, 0, 3], [2, -100, -100, -100, -100], [3, 3, 1, 0, -100], [-100, -100, -100, -100, -100]], "dtype": "int64"}, {"__nd__": [[0.036896679791633136, 0.7550597065611729, -0.00447814905519948, -0.3710777275995886], [0.238964649399828, -0.03829420646859828, -0.6270933309868154, -0.442533290725386], [0.883389658742294, 0.1771752976617628, 0.2081932675150889, -0.13827585916050578]], "dtype": "float64"}, {"__nd__": [-0.06897195637660919, 0.08916558155709342, -0.010462645267369168, -0.07591081117479302], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[0.1554638692112012, 0.06525308197947162, -0.01483475815304415, -0.20588219303762864], [-0.08649073155020122, -0.08467839483468392, -0.013647860813268596, 0.18481698719815376], [-0.05123750121232619, 0.10747914969929512, 0.09041012421725907, -0.14665177270422797]], "dtype": "float64"}, {"__nd__": [0.040533257128449955, 0.050976535200404856, 0.09914547115455895, -0.19065526348341377], "dtype": "float64"}, {"__nd__": [[0.15670377529776697, 0.0820531614666779, -0.004795594584473917, -0.23396134217997103], [-0.06534385447141869, -0.08840825585561629, -0.000414895120443598, 0.15416700544747863], [-0.059961864985401285, 0.15358115526291322, 0.08817948527082123, -0.18179877554833315]], "dtype": "float64"}, {"__nd__": [0.05181883526754181, 0.05207815017766623, 0.11198081205288188, -0.21587779749808989], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.18905338179353307, -0.5227484414807474, -0.41306354339189344], [-2.4414673826398556, 1.799707382720902, 1.1441658720372287], [-0.32542283686782436, 0.7738065867276614, 0.28121066979764925], [-0.5538228364240524, 0.9775674511260357, -0.31055654665915255], [-0.3288239040579627, -0.7921467553588982, 0.45495807124085547]], [[-0.09919805171738795, 0.5452887139646817, -0.6071856998706371], [0.12682784711186987, -0.8922740434297903, 0.8414649723701431], [0.18803508698068597, 0.33057100813532614, 0.41050391297026284], [-1.0107575001533344, 0.7831809961440773, 2.0567028183423686], [-1.6384425032355252, -1.7294114671544816, -1.50483141386432]], [[0.8414588934539998, 0.12871565747406846, 1.078342440739298], [0.722430872307499, 0.21057181237528058, 0.28403814525037085], [-0.16976049772313542, 0.8684602112115102, -1.1297159617807548], [-0.4218588261783162, 0.2429388530987352, 1.8014208584493328], [-0.7644641157203993, -1.0790604591369424, -0.5632871985040379]], [[0.9692722011381628, -0.23500550689182578, 1.324347019236957], [-1.872529517606024, 1.128523141900478, 1.0348662399295387], [-1.4186656898014849, 0.15358188653595775, 1.2157576351945363], [0.08791904230952238, 0.99970142215178, 2.3748869133171437], [0.2739322545923299, -0.2803823139429379, -0.7710521598195307]]], "dtype": "float64"}, {"__nd__": [[3, 0, 1, 0, 3], [2, -100, -100, -100, -100], [3, 3, 1, 0, -100], [-100, -100, -100, -100, -100]], "dtype": "int64"}, {"__nd__": [[0.036896679791633136, 0.7550597065611729, -0.00447814905519948, -0.3710777275995886], [0.238964649399828, -0.03829420646859828, -0.6270933309868154, -0.442533290725386], [0.883389658742294, 0.1771752976617628, 0.2081932675150889, -0.13827585916050578]], "dtype": "float64"}, {"__nd__": [-0.06897195637660919, 0.08916558155709342, -0.010462645267369168, -0.07591081117479302], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[0.1554638692112012, 0.06525308197947162, -0.014834758153044157, -0.20588219303762864], [-0.08649073155020122, -0.08467839483468394, -0.013647860813268596, 0.18481698719815381], [-0.05123750121232619, 0.10747914969929512, 0.09041012421725907, -0.14665177270422797]], "dtype": "float64"}, {"__nd__": [0.040533257128449955, 0.050976535200404856, 0.09914547115455895, -0.19065526348341377], "dtype": "float64"}, {"__nd__": [[0.08171459594980435, 0.03493598531156518, 0.010528358120038418, -0.12717893938140795], [-0.018344679949251506, -0.014747450058266044, -0.08970542941387663, 0.1227975594213942], [-0.05627638750698748, 0.03625988780073042, 0.14500661474944826, -0.12499011504319119]], "dtype": "float64"}, {"__nd__": [0.06764051420998618, 0.0887844269622795, -0.09964392397953001, -0.056781017192735705], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[2.0409191213851825, -2.5556650313141818, 0.41809884672577885, -0.5677696061279298], [-0.45264929211044586, -0.2155971630897659, -2.019986129147251, -0.23193237764418947], [-0.8652130762749417, 3.3229995166448827, 0.22578661322792176, -0.3526307943415954], [-0.2812874181513504, -0.6680463461089501, -1.0551505512051214, -0.39080097723465473], [0.48194538850678587, -0.2385536065733667, 0.9577587029597641, -0.19980212906658], [0.024259565076664623, 1.545820851212812, 0.5451055226876446, -0.505228735614018]], [[-0.1828389745977349, 0.5405251317548021, 1.9350880340988528, -0.2696203273419135], [-0.24355867907910456, 1.0023136012756912, -0.8864599431605871, -0.291720232439864], [0.8825389674564839, 0.5803500161908991, 0.09151670328235219, 0.6701043548284794], [-2.8281623068437627, 1.02130681750008, -0.9596447598081417, -1.6686198426559695], [0.27644575952099965, 0.7005448853493901, -0.4447674556827841, -1.0764058401008076], [0.026124833534033623, -0.05274730824287927, 1.4055981660180925, 0.7474079874793504]], [[0.19381564626462, 1.1116332052239921, -0.20552304990579248, -0.9258995736483681], [0.584058311025248, 0.5825384186556901, -0.2148289111268558, -0.7828085779639662], [0.22915390521326254, -2.4938942784579905, 0.690124770162812, 0.4913682607449912], [-1.6388571438904884, 0.06135350983817159, -0.9640996635412404, 0.7572210447581504], [-2.034167273443428, -0.9144945379945887, 0.7095799877420675, 1.156401048432157], [-2.158005380126208, -0.49803984475130336, 0.32802009254257697, -0.6092161379498706]], [[1.5906402313231438, -1.1912266816177808, 0.354531946286926, -1.0484055185445111], [1.4059629431348852, -0.021651229055558368, -0.3722505640006159, -1.7181849497326165], [1.6818255450666806, 0.7527785926973876, 0.753563837509362, 1.1378812589177814], [0.3492265781230293, -0.6392466105764212, -0.8002412270301018, -0.800199979361005], [1.3700723413337117, -1.4603812011954127, -0.5963695117707888, -0.32124391928619556], [0.22461902534909414, 0.5753493885078089, -1.2490970090955427, -1.730013451272522]], [[-0.004414232621967487, 1.2135638252860816, 0.7570580592965243, 0.21565078369996044], [-0.3171556440173552, 0.2932336958002246, -0.24333508574217566, 0.817206580492595], [-0.7944473388868819, 0.13423994708633882, -0.11078013611159404, 0.5433593895301524], [0.22463852364937692, 2.550034636307906, 1.498654758135483, 1.4967371655185107], [-2.0395038375946424, -0.3403166247023773, -0.6086106159129299, 0.5327215998890392], [-2.279026489055327, 1.1744986790091876, 1.0669833108953142, -1.3020708582457947]]], "dtype": "float64"}, {"__nd__": [[-100, -100, -100, -100, -100, -100], [-100, -100, -100, -100, -100, -100], [3, 4, 3, 2, 3, 4], [1, 4, 4, 0, 4, 3], [6, 2, 2, 4, 6, 0]], "dtype": "int64"}, {"__nd__": [[0.3288199519088864, -0.15257221288875805, -0.2262339435352007, 0.24233243910335076, -0.35074776508977673, -0.46529443810607773, 0.24063724613576615], [1.231566016052613, -0.12306677703240664, -0.27793289039677727, -0.5855784170127715, -0.6675054787796914, 0.262491617282113, 0.42540151638927626], [0.0045873604412451, 0.1662879943381042, 0.05795828875355198, 0.06932742302361042, -0.7630795262914237, -0.22905913652461057, 0.055739653794973865], [-0.3915833902818427, -0.23821487192990873, -0.40956009016386724, -0.16674833709922066, 0.42655413716256796, -0.20329017532779908, -0.07693534009855238]], "dtype": "float64"}, {"__nd__": [0.08137183594381181, 0.06447702141569163, 0.16952075589596333, -0.20904850781285883, 0.0856858944523106, -0.048228408677308224, 0.013469181881788084], "dtype": "float64"}, 3], "e": {"__tup__": [{"__nd__": [[0.15911636921582412, -0.124139315032428, 0.10635977655098061, 0.12665805867755303, -0.30834122132141417, -0.1002332000342396, 0.14057953194372394], [0.18504090650292362, 0.05126933073852266, -0.062245491154608916, 0.011986788360465076, -0.2030520806528492, 0.021467584928395667, -0.004467038722849018], [0.04841916377511462, -0.014451149787602622, 0.0709812778368, 0.0030781413443019584, -0.10989645469960382, -0.003243357947010715, 0.005112379478000573], [0.08691625995120071, 0.03388177194895687, -0.16414522351991234, 0.02467699594361998, 0.09497378318379118, -0.023692270390821737, -0.0526113171168347]], "dtype": "float64"}, {"__nd__": [0.1107454508677985, 0.05909122159543498, -0.026019994712006493, -0.11322817058577447, -0.15473523006914341, 0.12359603760402858, 0.0005506852996623179], "dtype": "float64"}, {"__nd__": [[0.12583466095181228, -0.08134953892366506, 0.06052931818281393, 0.05710393967495131, -0.19544484044638188, -0.08341317517399376, 0.11673963573446314], [0.1378281893448583, 0.03695688985226227, -0.03272534806267255, 0.009204903461395679, -0.17177382727479135, 0.02647594335467212, -0.005966750675724531], [0.04114820832042081, -0.0021187286280130377, 0.04976979373339608, 0.0019208057885988093, -0.10534464752563348, 0.0057550328103379515, 0.00886953550089285], [0.10907088842355633, 0.019728024229986256, -0.11858371142173876, 0.01566123947631433, 0.02165809242388248, -0.010355393664461528, -0.03717913946753919]], "dtype": "float64"}, {"__nd__": [0.07589021317682115, 0.047530353644439034, -0.049587518702534805, -0.04913408928914154, -0.07717253999223174, 0.0881330602721174, -0.035659479109469525], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.6517911526116896, -0.17471729232577715], [1.6637239913911968, 0.659147749832255], [-1.6413972945846467, -0.005203264171931977], [-0.6234637409883934, 0.14863152325202633]], [[-1.608187784186389, 0.2417718768768513], [0.23538091873745476, 1.5756260314314627], [0.3166450164719021, 0.5105466616976417], [-1.4931166849642326, 2.2527291247240275]], [[-1.9156455579583005, 1.1018018558224842], [-0.3299040738738497, -0.8806465179277969], [-0.6562850109290311, -0.6720146806586559], [0.3801890922489917, -0.11005869527805158]]], "dtype": "float64"}, {"__nd__": [[2, 2, 2, 2], [4, 2, 1, 4], [4, 1, 1, 1]], "dtype": "int64"}, {"__nd__": [[-0.17184717504974056, 0.10508646366805098, -0.7421342919981756, 0.49260779142909383, 0.0893567145591981], [0.503428385353542, 0.47959242933725554, -0.4899272831873384, -0.3988978789447691, -0.10166624288595129]], "dtype": "float64"}, {"__nd__": [0.07479597259631347, 0.08509672909755428, -0.07096848707086849, -0.06071971852553313, -0.07978229151665286], "dtype": "float64"}, 3], "e": {"__tup__": [{"__nd__": [[-0.16909232637840926, -0.05398750272248602, -0.1449594022363907, 0.0007902937467235086, 0.3672489375905625], [0.17790089427859104, 0.2352568343195987, -0.1622280388928387, 0.015506248606790129, -0.26643593831214113]], "dtype": "float64"}, {"__nd__": [0.25465309464468305, -0.1136018051832441, -0.1709475597867856, 0.13325001654514668, -0.10335374621980009], "dtype": "float64"}, {"__nd__": [[-0.16909232637840926, -0.05398750272248602, -0.1449594022363907, 0.0007902937467235086, 0.3672489375905625], [0.17790089427859104, 0.2352568343195987, -0.1622280388928387, 0.015506248606790129, -0.26643593831214113]], "dtype": "float64"}, {"__nd__": [0.25465309464468305, -0.1136018051832441, -0.1709475597867856, 0.13325001654514668, -0.10335374621980009], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.8019314252534474, -1.324358995628145, -0.24836162209524854, 0.4204452380655215, 1.1360465324896427, 0.10970639932180819], [-0.5526473205362324, -0.7847803553442784, 0.7487457707345911, 1.6347830429585775, 0.27276877584472176, -1.2333286640307717], [-0.9582652054360887, 1.6000190889991115, 0.2028824405086084, -1.7321348424395848, -0.08369619281702581, -1.1632259734447485], [-0.6292880940615545, -0.48800582327685743, -0.7133133716322436, 0.5533784703532895, -0.06308597192528916, -0.5894312580326048], [0.40963782655711695, 0.8298553070613239, -1.643023371405677, -0.256730126365494, -0.9807473560440125, -0.17315522486203205], [-1.2894187467538587, 0.0206903940375912, -0.03788574104406823, -0.304337750958489, -1.0479265051202462, -0.3961903304730927], [-1.091328901695709, -1.3552087462047395, 0.22478573245989314, -1.109349937891366, 1.1702961011782933, 0.7165876558738361], [-1.9978166924497212, 0.272128869412488, -1.1017166275810448, 0.033057220158269195, 0.04363199256942161, -1.9884297882311208], [-0.23342252376577002, -0.255790031399391, 0.9620005318430944, -1.1814468079562157, 0.7380418978456841, -1.0989727630364063]], [[-0.33129089269991674, -0.8404731684222111, 1.448731288921672, 0.5682130997882933, 2.4317325028452124, 0.6419163790823205], [0.8449927337191754, 0.8406828762653401, -0.6066115359095516, -0.07002844663838208, 1.350388867744626, -0.3965507651729716], [0.18879953129109867, -0.021223460417289796, 0.6092164928327407, -0.3649087419473726, -0.15236188875684828, 0.24238142867215043], [0.10302313848680769, -0.8649727462818201, 0.8957830431894438, -1.298481208246912, -1.2011155488035266, -1.282491794740462], [0.9669722789332148, -0.36060836889927, -0.9710363785210655, -1.1360213941896466, 0.42113113746240616, -1.054840662577835], [-1.2720782100976422, 0.6139930624688609, -1.1967077271925706, -0.32243815075139726, -0.006761540346521785, -0.4453353669298123], [-0.05409396159544828, 1.3387734990547715, -0.516894152589275, -1.2593071588782276, -1.8367457051936795, -0.20476611683386706], [-0.3522570104112305, 0.2650909069303235, -0.4642445916374816, -0.4786384630527245, -0.7213157478486046, -0.5197597723701788], [0.1602267063174243, -0.3803527272151388, 0.1004418528325065, 1.9011872000332568, 0.4791181507969435, -1.5760898454352261]], [[1.7335295856131068, 0.34781036998853543, -0.9414132864509847, 0.9070489576858102, 0.01765561090983897, -0.6152185330350691], [-0.6335089254336164, -0.9934317902518573, 0.048118983989838576, 1.0688166937495658, -0.3250520988063787, 0.42082412505635486], [1.875646053775279, -1.2145906329798497, 0.2575223239234917, -0.30643692685187435, -1.0592560998278446, -1.0258471994365972], [-0.0152851431653622, 0.4338634112001693, -0.531589820743032, 0.05439054435949247, 0.7222086544347099, 0.2965612469178253], [0.8686701897408925, 0.4613956081509272, 0.4880056756807095, 1.8266340649418156, 0.6231718736925668, 0.12947113235560923], [0.7099438034419987, -0.9191236755082103, -0.3377047857608698, 0.793220854194603, 0.6307867000938839, 1.5483616908312823], [0.010473028918682835, -1.4623277904204515, 1.9472469308762064, 1.0928928105587499, -1.0587374336035666, 1.3758236699684163], [0.03240279877908362, -1.813928769904867, -0.4203539214229468, -0.5089038339937678, 1.590848160322242, -0.7920686954195238], [-0.25361734037936806, -0.2767608068453714, -0.3789727629772628, -0.9140740610326185, 0.21908690902567676, 1.0768046464135546]], [[0.6239776511717521, -0.9274991160593102, -1.1498157571616898, 0.11897355358204866, -0.7065801636237695, -0.6301656488493008], [-1.6796945788401643, 1.9504916026999646, 0.9166191525408893, -0.9739073319069764, 0.9082102667945468, 1.3424871058246228], [-2.389553349676638, -0.548944785564265, -0.3879732181256381, 0.6482544703432906, -0.12146542230081914, -0.2304313358523204], [-0.05837138907215449, 1.8533257078420022, 2.159980469779012, -0.5248264370136562, -0.9262439932504336, 2.6925531473868567], [-0.9797446331927372, -0.5734020178922011, 0.036581509348738155, 0.4832462441452338, 1.029035549437811, 0.39097722167013954], [-0.8723830716951919, 0.5063823263497103, 0.24991757985122218, 1.876918992454176, -0.01499979815571943, -1.3369728999172332], [-1.0450199143839616, 1.450153058953162, -0.5401312070783946, -2.1044660100855603, -0.5806997054894565, 1.5099831293121058e-05], [1.1888306785373703, -1.014468137602427, 0.6666833259020761, 0.7952990996016167, -0.6993883083236738, -0.18758970531896946], [1.7694502363979239, 1.720484746826155, 0.8555220049018919, 0.33194635951240653, 1.1383096209632015, -0.14067728561527593]], [[-0.09509306569845781, -0.8603601395273159, 0.005563384668739004, -0.0820166249133826, 2.7736316673693673, -0.19284316578043667], [1.2708179900134384, 1.3204972743933365, -0.18647870191497867, 1.1693971923001296, -2.1768136494850046, 0.09492718485511631], [0.8572613994662929, -2.39786527267812, -1.159171198248098, 1.0560736173734917, -0.25701076898467345, -1.0981986938209587], [-0.3735956685207676, -0.5425883064344673, 0.7244473133352144, 0.4538858794596457, -0.2802283393557666, -0.6653426856286421], [-0.0538756638943387, 1.325155927345223, 0.3490261125780868, 0.6389455936428303, -0.1506815972085555, -1.2918162500762125], [-0.7108435355871252, 0.6512365658972803, 0.10321610258141592, -0.9038398688268016, -1.617870425602848, 0.11068190035142705], [-0.339531008698164, -0.350530218612727, -0.3294499574610804, 0.28390511348310316, 0.29444772495280114, 0.9621285502204014], [0.9284215804413863, 1.3320930804934341, 0.8063630567820633, -0.3356080333276904, -0.05426894624359061, 0.4942150948314312], [-0.31029227000142345, 0.5494387802798892, 1.1536094769949883, -0.8012460099660716, -1.770384050846499, 1.1936549552915572]], [[0.01775991033501458, 0.7086698231113021, -1.3323053543858547, 1.7329925604781644, 0.7386198894684642, -0.12607417369146828], [-0.3519880388050872, 0.11389588045524304, -0.16673134056534972, -0.2786901272438552, -1.1195999755255746, 1.262809438921045], [-0.039577684569559124, 0.07463563738960281, 0.09291598860047003, -2.358217336741871, 0.5271414581243252, -0.32915493062535894], [-0.2892284645923461, 0.06978787257481246, -0.9724756053284209, 0.8433808099877073, -0.6958646238070096, -0.9036580547128793], [-0.8621896009823745, 0.15875269513992676, 0.7457794743624686, -0.5950532355462611, -1.510186675915287, 1.062490693153603], [-1.0650835950433684, 1.194830368529916, -0.05429587697711575, -0.6983498940240875, -0.6400399491357172, 0.4618661736925272], [0.8893717949967992, -0.19817022758122477, 0.946934382371629, 2.08697820766597, -0.20830834526001082, -1.1494466412305686], [-1.0544003485304514, -1.3903353550011621, -0.639665450940511, -0.7237792369318543, -1.3251033289662522, 0.5301842547135], [-2.6113390049731797, -1.3207989716931114, 2.0152518061838767, 1.0378024284634166, 0.5227916833432323, 0.45725905895534386]], [[1.7709371728004815, 0.5356296362853685, 0.0775770872272132, -1.2301601607845671, -0.07366681732892769, -0.7338839983148214], [-0.7061752749030227, -0.7142389333341079, -0.04464686469247794, -0.32971081974207533, 0.8711105109344268, -1.3261814419117322], [-0.16947019674286637, 1.8594974964345012, 0.0388973371949769, 1.198615221448642, 1.1368634972033143, 1.3166478203373606], [0.7381191695252466, -1.3804460312528912, -0.2673877057870027, -0.38033558246862037, -1.5488553043334206, -0.4638816211751379], [-1.2500798351337994, -1.6655514460193832, 0.21089625656145172, -0.04359127244566091, -0.6878149095751852, -0.9505863626128569], [-0.541273442072544, -0.39409036937385966, -0.4081102825826052, -0.052707747928468524, -0.43554067396098334, 0.3869719823711992], [0.5228708948391112, 0.35786967764803124, -0.6631035145259176, -0.6165061466820784, 0.4348641034709542, -0.197042778476594], [-0.04315170860457781, -2.4919952503112044, -2.700103979886236, 0.33714819206874613, 0.3295586563507561, 1.3934318714435818], [0.7211631188958838, 0.22173880016804176, 0.01636245608868217, -1.426740428209398, 0.8831758896325703, -0.6035860153728927]]], "dtype": "float64"}, {"__nd__": [[2, 10, 0, 0, 9, 2, -100, -100, -100], [6, 6, 8, 7, 6, 9, 5, 0, 2], [0, -100, -100, -100, -100, -100, -100, -100, -100], [1, 7, 3, 6, 3, 5, 4, 4, 1], [2, 7, 9, 9, 6, 0, 8, 10, -100], [-100, -100, -100, -100, -100, -100, -100, -100, -100], [6, 8, 6, 9, 1, 2, 1, -100, -100]], "dtype": "int64"}, {"__nd__": [[-0.11164779988610311, 0.3620096149148162, 0.45881247885492465, 0.46630620157204977, 0.10627549280538835, -0.233194046552954, -0.09604366458876987, 0.5547998557573296, 0.7199590574786171, -0.17065619504305177, -0.23865200127222688], [0.6237748349738497, 0.3618245115667212, 0.34862909501265416, -0.6561668314766673, -0.1868787207331968, -0.2574977646020882, 0.10395232952234879, -0.8424709898654748, 0.6358764040669388, 0.15006589767527886, 0.21741851273989893], [-0.5920701884424364, -0.013210185684084403, 0.6673251440784499, -0.7848478012294603, 0.1528840286887781, 0.3494024571571941, -0.34597355154360127, 0.40737061413428655, -0.16503451528207796, 0.6148832765157783, 0.26375588771257624], [0.11218294286419457, 1.004457990707041, 0.4159482291870472, 0.6543178543241671, -0.5947598361152727, -0.3233035338593377, 0.41828450146588364, 0.45280593166588023, -0.02787001499506506, 0.38986326711283525, -0.5291768730732942], [0.10691761286018961, -0.5736419324689233, 0.17028526688905224, -0.2430859535699963, -0.3204987583221377, 0.028149515091318446, 0.9087300318473212, -0.5058808672277085, -0.3008753080709376, 0.5338320728676305, -0.21178333499933555], [0.8846321386048187, 0.11126485192318987, 0.045315957075526735, 0.17844206943143942, 0.8705897982577016, 0.32768076724533496, 0.25835606645440323, 0.1259288815750003, -0.7812094166502325, -0.4426360140725756, 0.7416720605186317]], "dtype": "float64"}, {"__nd__": [0.08218783319217637, 0.05060648618402574, 0.06370627277466291, -0.05435930521122606, 0.03379642676721119, -0.0900794630349013, 0.03913623213601605, -0.07141351107069187, -0.021381982915618124, 0.04534148448314499, -0.032678770337045644], "dtype": "float64"}, 3], "e": {"__tup__": [{"__nd__": [[-0.010145339315900531, -0.02537488049158935, 0.08066871641616222, 0.10229742584865838, -0.021737202155855076, -0.007845123104367767, -0.10520666955736428, 0.018458494248009376, 0.07074337074065983, -0.047571203975422346, -0.05428758865299051], [0.0016091532918433554, 0.0420168646633568, 0.10389455824319899, -0.04073917780109511, 0.013448752684382684, -0.05199199921200156, -0.14342077752986737, -0.1230760589362467, 0.08924581024050675, 0.059605877595947626, 0.04940699675997453], [0.03063354934857874, 0.010000829509621017, 0.03605811824217055, -0.041619375273130116, 0.005397800715755987, 0.009542637971090832, -0.06728194675360705, -0.03972301912991557, -0.03431701596617031, 0.10910824320826731, -0.017799821872661385], [0.024518434618269268, 0.05238043638454078, -0.03343289890288371, 0.004059363602100224, -0.0035102913994363777, -0.030426287646863937, 0.035676764933726325, 0.04514186189631915, -0.031312891386919686, 0.024311407537704027, -0.08740589963655607], [0.0646186633617017, -0.04699512483850102, -0.054746648086813954, -0.04448920567217773, 0.0009550360211932653, 0.04266472043602698, -0.016963691267135622, 0.025982392435164016, -0.0601120135203118, 0.12451132016820252, -0.03542544903734836], [0.08719826798451964, 0.013533630313801078, 0.025625616894346462, -0.05760005351353895, 0.01196475697249539, 0.0234532151383574, -0.04189155978197648, -0.05004440176267817, -0.07188520824957695, 0.01577049836865849, 0.04387523763559213]], "dtype": "float64"}, {"__nd__": [-0.03548951075863786, -0.008438906273983953, -0.04861434235245726, 0.04761538641471563, 0.02292874511629735, 0.013635210499487597, -0.07553091578460705, 0.010842878669904556, 0.054925810134497245, -0.020185512885831567, 0.03831115722061531], "dtype": "float64"}, {"__nd__": [[-0.0092121993354795, 0.000690221176780359, 0.08369837057718821, 0.07972946377477431, -0.015931184473439223, -0.012054483259348533, -0.13322239688462975, 0.012880980066942964, 0.10229680526598028, -0.0656558625362077, -0.04321971437256144], [0.018262051706325916, 0.06981179795474554, 0.09805273670524735, -0.04794005472674116, 0.0041253452756679215, -0.05290518831644583, -0.1728675780261317, -0.1236095550302734, 0.09464641822818187, 0.07457581058302243, 0.037848215646401076], [0.021588102094348676, 0.017876188170062514, 0.038421152847146964, -0.037534815637503396, 0.0006080357003403723, 0.006390180274838603, -0.05907679445939351, -0.034928575030324746, -0.03138239127416686, 0.09574715456086283, -0.017708237246211456], [0.02827733883176229, 0.060175371782854856, -0.029008456995309774, -0.0018514996634970485, -0.008622675048068365, -0.026381080976094047, 0.03272635867570822, 0.029955010383888148, -0.03908386791101342, 0.027797640422766313, -0.07398413950299718], [0.06331698348397774, -0.030479132143873718, -0.027579095637545757, -0.04575871693622727, -0.0049507102720929615, 0.033168017244904854, -0.036110631323979875, 0.0014601709673950566, -0.07327325353885297, 0.14985982380500884, -0.029653455648713932], [0.08365597059377736, 0.04206892810821895, 0.00925241938502596, -0.054517070572627606, 0.00439972911253893, 0.009798179709560723, -0.04838297788231197, -0.05298796888573759, -0.03810587199799825, 0.011645033238110754, 0.033173629191442754]], "dtype": "float64"}, {"__nd__": [-0.01020473079211949, -0.04905403664171201, -0.05579325796968641, 0.05848586768038952, 0.03436763235644069, 0.025541882238394972, -0.09830295774361257, 0.03339316046490311, 0.04486671856070766, -0.025455964376180827, 0.0421556862224753], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.8019314252534474, -1.324358995628145, -0.24836162209524854, 0.4204452380655215, 1.1360465324896427, 0.10970639932180819], [-0.5526473205362324, -0.7847803553442784, 0.7487457707345911, 1.6347830429585775, 0.27276877584472176, -1.2333286640307717], [-0.9582652054360887, 1.6000190889991115, 0.2028824405086084, -1.7321348424395848, -0.08369619281702581, -1.1632259734447485], [-0.6292880940615545, -0.48800582327685743, -0.7133133716322436, 0.5533784703532895, -0.06308597192528916, -0.5894312580326048], [0.40963782655711695, 0.8298553070613239, -1.643023371405677, -0.256730126365494, -0.9807473560440125, -0.17315522486203205], [-1.2894187467538587, 0.0206903940375912, -0.03788574104406823, -0.304337750958489, -1.0479265051202462, -0.3961903304730927], [-1.091328901695709, -1.3552087462047395, 0.22478573245989314, -1.109349937891366, 1.1702961011782933, 0.7165876558738361], [-1.9978166924497212, 0.272128869412488, -1.1017166275810448, 0.033057220158269195, 0.04363199256942161, -1.9884297882311208], [-0.23342252376577002, -0.255790031399391, 0.9620005318430944, -1.1814468079562157, 0.7380418978456841, -1.0989727630364063]], [[-0.33129089269991674, -0.8404731684222111, 1.448731288921672, 0.5682130997882933, 2.4317325028452124, 0.6419163790823205], [0.8449927337191754, 0.8406828762653401, -0.6066115359095516, -0.07002844663838208, 1.350388867744626, -0.3965507651729716], [0.18879953129109867, -0.021223460417289796, 0.6092164928327407, -0.3649087419473726, -0.15236188875684828, 0.24238142867215043], [0.10302313848680769, -0.8649727462818201, 0.8957830431894438, -1.298481208246912, -1.2011155488035266, -1.282491794740462], [0.9669722789332148, -0.36060836889927, -0.9710363785210655, -1.1360213941896466, 0.42113113746240616, -1.054840662577835], [-1.2720782100976422, 0.6139930624688609, -1.1967077271925706, -0.32243815075139726, -0.006761540346521785, -0.4453353669298123], [-0.05409396159544828, 1.3387734990547715, -0.516894152589275, -1.2593071588782276, -1.8367457051936795, -0.20476611683386706], [-0.3522570104112305, 0.2650909069303235, -0.4642445916374816, -0.4786384630527245, -0.7213157478486046, -0.5197597723701788], [0.1602267063174243, -0.3803527272151388, 0.1004418528325065, 1.9011872000332568, 0.4791181507969435, -1.5760898454352261]], [[1.7335295856131068, 0.34781036998853543, -0.9414132864509847, 0.9070489576858102, 0.01765561090983897, -0.6152185330350691], [-0.6335089254336164, -0.9934317902518573, 0.048118983989838576, 1.0688166937495658, -0.3250520988063787, 0.42082412505635486], [1.875646053775279, -1.2145906329798497, 0.2575223239234917, -0.30643692685187435, -1.0592560998278446, -1.0258471994365972], [-0.0152851431653622, 0.4338634112001693, -0.531589820743032, 0.05439054435949247, 0.7222086544347099, 0.2965612469178253], [0.8686701897408925, 0.4613956081509272, 0.4880056756807095, 1.8266340649418156, 0.6231718736925668, 0.12947113235560923], [0.7099438034419987, -0.9191236755082103, -0.3377047857608698, 0.793220854194603, 0.6307867000938839, 1.5483616908312823], [0.010473028918682835, -1.4623277904204515, 1.9472469308762064, 1.0928928105587499, -1.0587374336035666, 1.3758236699684163], [0.03240279877908362, -1.813928769904867, -0.4203539214229468, -0.5089038339937678, 1.590848160322242, -0.7920686954195238], [-0.25361734037936806, -0.2767608068453714, -0.3789727629772628, -0.9140740610326185, 0.21908690902567676, 1.0768046464135546]], [[0.6239776511717521, -0.9274991160593102, -1.1498157571616898, 0.11897355358204866, -0.7065801636237695, -0.6301656488493008], [-1.6796945788401643, 1.9504916026999646, 0.9166191525408893, -0.9739073319069764, 0.9082102667945468, 1.3424871058246228], [-2.389553349676638, -0.548944785564265, -0.3879732181256381, 0.6482544703432906, -0.12146542230081914, -0.2304313358523204], [-0.05837138907215449, 1.8533257078420022, 2.159980469779012, -0.5248264370136562, -0.9262439932504336, 2.6925531473868567], [-0.9797446331927372, -0.5734020178922011, 0.036581509348738155, 0.4832462441452338, 1.029035549437811, 0.39097722167013954], [-0.8723830716951919, 0.5063823263497103, 0.24991757985122218, 1.876918992454176, -0.01499979815571943, -1.3369728999172332], [-1.0450199143839616, 1.450153058953162, -0.5401312070783946, -2.1044660100855603, -0.5806997054894565, 1.5099831293121058e-05], [1.1888306785373703, -1.014468137602427, 0.6666833259020761, 0.7952990996016167, -0.6993883083236738, -0.18758970531896946], [1.7694502363979239, 1.720484746826155, 0.8555220049018919, 0.33194635951240653, 1.1383096209632015, -0.14067728561527593]], [[-0.09509306569845781, -0.8603601395273159, 0.005563384668739004, -0.0820166249133826, 2.7736316673693673, -0.19284316578043667], [1.2708179900134384, 1.3204972743933365, -0.18647870191497867, 1.1693971923001296, -2.1768136494850046, 0.09492718485511631], [0.8572613994662929, -2.39786527267812, -1.159171198248098, 1.0560736173734917, -0.25701076898467345, -1.0981986938209587], [-0.3735956685207676, -0.5425883064344673, 0.7244473133352144, 0.4538858794596457, -0.2802283393557666, -0.6653426856286421], [-0.0538756638943387, 1.325155927345223, 0.3490261125780868, 0.6389455936428303, -0.1506815972085555, -1.2918162500762125], [-0.7108435355871252, 0.6512365658972803, 0.10321610258141592, -0.9038398688268016, -1.617870425602848, 0.11068190035142705], [-0.339531008698164, -0.350530218612727, -0.3294499574610804, 0.28390511348310316, 0.29444772495280114, 0.9621285502204014], [0.9284215804413863, 1.3320930804934341, 0.8063630567820633, -0.3356080333276904, -0.05426894624359061, 0.4942150948314312], [-0.31029227000142345, 0.5494387802798892, 1.1536094769949883, -0.8012460099660716, -1.770384050846499, 1.1936549552915572]], [[0.01775991033501458, 0.7086698231113021, -1.3323053543858547, 1.7329925604781644, 0.7386198894684642, -0.12607417369146828], [-0.3519880388050872, 0.11389588045524304, -0.16673134056534972, -0.2786901272438552, -1.1195999755255746, 1.262809438921045], [-0.039577684569559124, 0.07463563738960281, 0.09291598860047003, -2.358217336741871, 0.5271414581243252, -0.32915493062535894], [-0.2892284645923461, 0.06978787257481246, -0.9724756053284209, 0.8433808099877073, -0.6958646238070096, -0.9036580547128793], [-0.8621896009823745, 0.15875269513992676, 0.7457794743624686, -0.5950532355462611, -1.510186675915287, 1.062490693153603], [-1.0650835950433684, 1.194830368529916, -0.05429587697711575, -0.6983498940240875, -0.6400399491357172, 0.4618661736925272], [0.8893717949967992, -0.19817022758122477, 0.946934382371629, 2.08697820766597, -0.20830834526001082, -1.1494466412305686], [-1.0544003485304514, -1.3903353550011621, -0.639665450940511, -0.7237792369318543, -1.3251033289662522, 0.5301842547135], [-2.6113390049731797, -1.3207989716931114, 2.0152518061838767, 1.0378024284634166, 0.5227916833432323, 0.45725905895534386]], [[1.7709371728004815, 0.5356296362853685, 0.0775770872272132, -1.2301601607845671, -0.07366681732892769, -0.7338839983148214], [-0.7061752749030227, -0.7142389333341079, -0.04464686469247794, -0.32971081974207533, 0.8711105109344268, -1.3261814419117322], [-0.16947019674286637, 1.8594974964345012, 0.0388973371949769, 1.198615221448642, 1.1368634972033143, 1.3166478203373606], [0.7381191695252466, -1.3804460312528912, -0.2673877057870027, -0.38033558246862037, -1.5488553043334206, -0.4638816211751379], [-1.2500798351337994, -1.6655514460193832, 0.21089625656145172, -0.04359127244566091, -0.6878149095751852, -0.9505863626128569], [-0.541273442072544, -0.39409036937385966, -0.4081102825826052, -0.052707747928468524, -0.43554067396098334, 0.3869719823711992], [0.5228708948391112, 0.35786967764803124, -0.6631035145259176, -0.6165061466820784, 0.4348641034709542, -0.197042778476594], [-0.04315170860457781, -2.4919952503112044, -2.700103979886236, 0.33714819206874613, 0.3295586563507561, 1.3934318714435818], [0.7211631188958838, 0.22173880016804176, 0.01636245608868217, -1.426740428209398, 0.8831758896325703, -0.6035860153728927]]], "dtype": "float64"}, {"__nd__": [[2, 10, 0, 0, 9, 2, -100, -100, -100], [6, 6, 8, 7, 6, 9, 5, 0, 2], [0, -100, -100, -100, -100, -100, -100, -100, -100], [1, 7, 3, 6, 3, 5, 4, 4, 1], [2, 7, 9, 9, 6, 0, 8, 10, -100], [-100, -100, -100, -100, -100, -100, -100, -100, -100], [6, 8, 6, 9, 1, 2, 1, -100, -100]], "dtype": "int64"}, {"__nd__": [[-0.11164779988610311, 0.3620096149148162, 0.45881247885492465, 0.46630620157204977, 0.10627549280538835, -0.233194046552954, -0.09604366458876987, 0.5547998557573296, 0.7199590574786171, -0.17065619504305177, -0.23865200127222688], [0.6237748349738497, 0.3618245115667212, 0.34862909501265416, -0.6561668314766673, -0.1868787207331968, -0.2574977646020882, 0.10395232952234879, -0.8424709898654748, 0.6358764040669388, 0.15006589767527886, 0.21741851273989893], [-0.5920701884424364, -0.013210185684084403, 0.6673251440784499, -0.7848478012294603, 0.1528840286887781, 0.3494024571571941, -0.34597355154360127, 0.40737061413428655, -0.16503451528207796, 0.6148832765157783, 0.26375588771257624], [0.11218294286419457, 1.004457990707041, 0.4159482291870472, 0.6543178543241671, -0.5947598361152727, -0.3233035338593377, 0.41828450146588364, 0.45280593166588023, -0.02787001499506506, 0.38986326711283525, -0.5291768730732942], [0.10691761286018961, -0.5736419324689233, 0.17028526688905224, -0.2430859535699963, -0.3204987583221377, 0.028149515091318446, 0.9087300318473212, -0.5058808672277085, -0.3008753080709376, 0.5338320728676305, -0.21178333499933555], [0.8846321386048187, 0.11126485192318987, 0.045315957075526735, 0.17844206943143942, 0.8705897982577016, 0.32768076724533496, 0.25835606645440323, 0.1259288815750003, -0.7812094166502325, -0.4426360140725756, 0.7416720605186317]], "dtype": "float64"}, {"__nd__": [0.08218783319217637, 0.05060648618402574, 0.06370627277466291, -0.05435930521122606, 0.03379642676721119, -0.0900794630349013, 0.03913623213601605, -0.07141351107069187, -0.021381982915618124, 0.04534148448314499, -0.032678770337045644], "dtype": "float64"}, 7], "e": {"__tup__": [{"__nd__": [[-0.010145339315900528, -0.02537488049158936, 0.08066871641616224, 0.10229742584865838, -0.02173720215585508, -0.007845123104367769, -0.10520666955736428, 0.018458494248009376, 0.07074337074065983, -0.04757120397542233, -0.0542875886529905], [0.0016091532918433554, 0.0420168646633568, 0.10389455824319899, -0.04073917780109511, 0.01344875268438268, -0.05199199921200156, -0.14342077752986734, -0.12307605893624672, 0.08924581024050675, 0.059605877595947626, 0.04940699675997452], [0.03063354934857874, 0.01000082950962102, 0.03605811824217055, -0.04161937527313013, 0.005397800715755987, 0.009542637971090832, -0.06728194675360705, -0.03972301912991556, -0.03431701596617031, 0.10910824320826729, -0.01779982187266138], [0.024518434618269264, 0.05238043638454078, -0.0334328989028837, 0.004059363602100224, -0.0035102913994363855, -0.030426287646863944, 0.035676764933726325, 0.04514186189631914, -0.03131289138691967, 0.024311407537704013, -0.08740589963655605], [0.06461866336170169, -0.04699512483850101, -0.05474664808681397, -0.04448920567217773, 0.0009550360211932653, 0.04266472043602698, -0.01696369126713565, 0.02598239243516403, -0.0601120135203118, 0.12451132016820252, -0.03542544903734836], [0.08719826798451963, 0.013533630313801078, 0.025625616894346455, -0.05760005351353897, 0.01196475697249539, 0.0234532151383574, -0.04189155978197646, -0.05004440176267817, -0.07188520824957695, 0.015770498368658487, 0.043875237635592125]], "dtype": "float64"}, {"__nd__": [-0.03548951075863786, -0.00843890627398396, -0.04861434235245726, 0.04761538641471562, 0.02292874511629735, 0.013635210499487599, -0.07553091578460705, 0.010842878669904552, 0.05492581013449724, -0.020185512885831567, 0.03831115722061531], "dtype": "float64"}, {"__nd__": [[-0.2041995967026612, 0.028987050277304787, 0.08593015807767466, 0.1100758179895951, -0.013099056549571931, -0.007076714802701239, -0.06500102850705236, 0.02097008130764138, 0.11414336622988629, -0.036803198449272795, -0.03392687887084278], [-0.0464652337507352, 0.042590551304737545, 0.0840853579701842, -0.023825296728448596, 0.008948717919790485, -0.0344931450403875, -0.1049439936061585, -0.08544584847487639, 0.08070371621558314, 0.041382957430353105, 0.03746221675995775], [0.13372169868488648, -0.016532942531346254, 0.018343736602176336, -0.05681210056724381, 0.0005587030569194516, 0.004323472113276199, -0.052801838614423904, -0.033282188095327875, -0.058818025109795356, 0.08190447307535294, -0.020604988614474167], [-0.08274742198340124, 0.06155914251438028, -0.012369125144433807, 0.029639953149111373, -0.002752969374806955, -0.019398475453220466, 0.030723344239995728, 0.03560860434134299, 0.007112411500791591, 0.022829930777080544, -0.07020539456684005], [0.04305005341143051, -0.032422362398845224, -0.03845427811108097, -0.030232248083747124, -0.0018235026559430471, 0.02633552416236276, -0.008641887529989288, 0.01751261072078069, -0.04626305253328234, 0.09785576587029379, -0.026916622851979773], [0.14419187362107636, -0.004017417795315049, 0.008841984829674375, -0.05786217754617291, 0.00483609308844769, 0.011041822462836626, -0.03305330114771057, -0.04203165802740934, -0.07192849305163326, 0.006518836618247095, 0.033462436947959015]], "dtype": "float64"}, {"__nd__": [-0.15286150366552106, 0.018454670745633596, -0.037674181579943455, 0.0674501742746613, 0.022464014342576274, 0.016520799985842004, -0.041886474696516546, 0.021701665803415503, 0.07776516895008603, -0.014269772024446183, 0.022335437864212517], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(accumulate_grads, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")